In [2]:
from google.colab import files
uploaded = files.upload()


Saving olist_customers_dataset.xlsx to olist_customers_dataset.xlsx
Saving olist_order_items_dataset.xlsx to olist_order_items_dataset.xlsx
Saving olist_order_payments_dataset.xlsx to olist_order_payments_dataset.xlsx
Saving olist_order_reviews_dataset.xlsx to olist_order_reviews_dataset.xlsx
Saving olist_orders_dataset.xlsx to olist_orders_dataset.xlsx
Saving olist_products_dataset.xlsx to olist_products_dataset.xlsx
Saving olist_sellers_dataset.xlsx to olist_sellers_dataset.xlsx
Saving product_category_name_translation.xlsx to product_category_name_translation (1).xlsx


In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

table_map = {
    'olist_customers_dataset': 'customers',
    'olist_orders_dataset': 'orders',
    'olist_order_items_dataset': 'order_items',
    'olist_order_payments_dataset': 'payments',
    'olist_order_reviews_dataset': 'reviews',
    'olist_products_dataset': 'products',
    'olist_sellers_dataset': 'sellers',
    'product_category_name_translation': 'category_translation'
}

dataframes = {}

for file_name in uploaded.keys():
    file_lower = file_name.lower()

    table_name = None
    for key, value in table_map.items():
        if key in file_lower:
            table_name = value
            break

    if table_name is None:
        print("Skipped:", file_name)
        continue

    df = pd.read_excel(file_name)
    df.columns = df.columns.str.strip()

    dataframes[table_name] = df
    df.to_sql(table_name, conn, index=False, if_exists='replace')

    print(f"{table_name} loaded successfully:", df.shape)

customers loaded successfully: (99441, 5)
order_items loaded successfully: (112650, 7)
payments loaded successfully: (103886, 5)
reviews loaded successfully: (99224, 7)
orders loaded successfully: (99441, 8)
products loaded successfully: (32951, 9)
sellers loaded successfully: (3095, 4)
category_translation loaded successfully: (71, 2)


In [4]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

In [5]:
run_sql("""
SELECT
    ROUND(SUM(oi.price), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    COUNT(DISTINCT oi.seller_id) AS total_sellers,
    COUNT(DISTINCT oi.product_id) AS total_products,
    ROUND(SUM(oi.price) * 1.0 / COUNT(DISTINCT o.order_id), 2) AS average_order_value
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id;
""")

,total_revenue,total_orders,total_customers,total_sellers,total_products,average_order_value
0,13591643.7,99441,96096,3095,32951,136.68


In [6]:
run_sql("""
SELECT
    strftime('%Y', o.order_purchase_timestamp) AS order_year,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT oi.order_id) AS item_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY order_year
ORDER BY order_year;
""")

,order_year,total_orders,item_orders,total_revenue
0,2016,329,312,49785.92
1,2017,45101,44579,6155806.98
2,2018,54011,53775,7386050.80


In [8]:
run_sql("""
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT oi.order_id) AS item_orders,
    ROUND(COALESCE(SUM(oi.price), 0), 2) AS total_revenue,
    ROUND(
        COALESCE(SUM(oi.price), 0) * 1.0 / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2
    ) AS average_order_value
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY order_month
ORDER BY order_month;
""")

,order_month,total_orders,item_orders,total_revenue,average_order_value
0,2016-09,4,3,267.36,66.84
1,2016-10,324,308,49507.66,152.80
2,2016-12,1,1,10.90,10.90
3,2017-01,800,789,120312.87,150.39
4,2017-02,1780,1733,247303.02,138.93
5,2017-03,2682,2641,374344.30,139.58
6,2017-04,2404,2391,359927.23,149.72
7,2017-05,3700,3660,506071.14,136.78
8,2017-06,3245,3217,433038.60,133.45
9,2017-07,4026,3969,498031.48,123.70


In [9]:
run_sql("""
SELECT
    REPLACE(ct.product_category_name_english, '_', ' ') AS category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category_name
ORDER BY total_revenue DESC
LIMIT 10;
""")

,category_name,total_orders,total_items_sold,total_revenue
0,health beauty,8836,9670,1258681.34
1,watches gifts,5624,5991,1205005.68
2,bed bath table,9417,11115,1036988.68
3,sports leisure,7720,8641,988048.97
4,computers accessories,6689,7827,911954.32
5,furniture decor,6449,8334,729762.49
6,cool stuff,3632,3796,635290.85
7,housewares,5884,6964,632248.66
8,auto,3897,4235,592720.11
9,garden tools,3518,4347,485256.46


In [10]:
run_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    ROUND(COALESCE(SUM(oi.price), 0), 2) AS total_revenue,
    ROUND(
        COALESCE(SUM(oi.price), 0) * 1.0 / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2
    ) AS average_order_value
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
""")

,customer_state,total_orders,total_customers,total_revenue,average_order_value
0,SP,41746,40302,5202955.05,124.63
1,RJ,12852,12384,1824092.67,141.93
2,MG,11635,11259,1585308.03,136.25
3,RS,5466,5277,750304.02,137.27
4,PR,5045,4882,683083.76,135.40
5,SC,3637,3534,520553.34,143.13
6,BA,3380,3277,511349.99,151.29
7,DF,2140,2075,302603.94,141.40
8,GO,2020,1952,294591.95,145.84
9,ES,2033,1964,275037.31,135.29


In [11]:
run_sql("""
SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(
        SUM(payment_value) * 100.0 / (SELECT SUM(payment_value) FROM payments),
        2
    ) AS payment_percentage
FROM payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
""")

,payment_type,total_orders,total_payment_value,payment_percentage
0,credit_card,76505,12542084.19,78.34
1,boleto,19784,2869361.27,17.92
2,voucher,3866,379436.87,2.37
3,debit_card,1528,217989.79,1.36
4,not_defined,3,0.00,0.00


In [12]:
run_sql("""
SELECT
    CASE
        WHEN order_delivered_customer_date IS NULL THEN 'Not Delivered'
        WHEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) > 0 THEN 'Delayed'
        ELSE 'On Time / Early'
    END AS delivery_status,

    COUNT(DISTINCT order_id) AS total_orders,

    ROUND(
        AVG(
            CASE
                WHEN order_delivered_customer_date IS NOT NULL
                THEN julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)
            END
        ),
        2
    ) AS avg_delivery_days,

    ROUND(
        AVG(
            CASE
                WHEN order_delivered_customer_date IS NOT NULL
                THEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date)
            END
        ),
        2
    ) AS avg_delay_days

FROM orders
GROUP BY delivery_status
ORDER BY total_orders DESC;
""")

,delivery_status,total_orders,avg_delivery_days,avg_delay_days
0,On Time / Early,88649,10.88,-13.01
1,Delayed,7827,31.52,9.55
2,Not Delivered,2965,NaN,NaN


In [13]:
run_sql("""
WITH state_delivery AS (
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS total_orders,

        COUNT(DISTINCT CASE
            WHEN o.order_status = 'delivered'
            THEN o.order_id
        END) AS delivered_orders,

        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date IS NOT NULL
             AND julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN o.order_id
        END) AS delayed_orders,

        ROUND(
            AVG(
                CASE
                    WHEN o.order_delivered_customer_date IS NOT NULL
                    THEN julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)
                END
            ),
            2
        ) AS avg_delivery_days
    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
)

SELECT
    customer_state,
    total_orders,
    delivered_orders,
    delayed_orders,
    avg_delivery_days,
    ROUND(delayed_orders * 100.0 / NULLIF(delivered_orders, 0), 2) AS delay_rate_percentage
FROM state_delivery
ORDER BY delay_rate_percentage DESC
LIMIT 10;
""")

,customer_state,total_orders,delivered_orders,delayed_orders,avg_delivery_days,delay_rate_percentage
0,AL,413,397,95,24.54,23.93
1,MA,747,717,141,21.57,19.67
2,PI,495,476,76,19.46,15.97
3,CE,1336,1279,196,21.27,15.32
4,SE,350,335,51,21.52,15.22
5,BA,3380,3256,457,19.34,14.04
6,RJ,12852,12350,1664,15.31,13.47
7,TO,280,274,35,17.66,12.77
8,PA,975,946,117,23.77,12.37
9,ES,2033,1995,244,15.79,12.23


In [14]:
run_sql("""
WITH state_delivery AS (
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS total_orders,

        COUNT(DISTINCT CASE
            WHEN o.order_status = 'delivered'
            THEN o.order_id
        END) AS delivered_orders,

        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date IS NOT NULL
             AND CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS INTEGER) > 0
            THEN o.order_id
        END) AS delayed_orders,

        ROUND(
            AVG(
                CASE
                    WHEN o.order_delivered_customer_date IS NOT NULL
                    THEN CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS INTEGER)
                END
            ),
            2
        ) AS avg_delivery_days
    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
)

SELECT
    customer_state,
    total_orders,
    delivered_orders,
    delayed_orders,
    avg_delivery_days,
    ROUND(delayed_orders * 100.0 / NULLIF(delivered_orders, 0), 2) AS delay_rate_percentage
FROM state_delivery
ORDER BY delay_rate_percentage DESC
LIMIT 10;
""")


,customer_state,total_orders,delivered_orders,delayed_orders,avg_delivery_days,delay_rate_percentage
0,AL,413,397,85,24.04,21.41
1,MA,747,717,125,21.12,17.43
2,SE,350,335,51,21.03,15.22
3,PI,495,476,66,18.99,13.87
4,CE,1336,1279,176,20.82,13.76
5,RR,46,41,5,28.98,12.20
6,BA,3380,3256,396,18.87,12.16
7,RJ,12852,12350,1495,14.85,12.11
8,PA,975,946,106,23.32,11.21
9,ES,2033,1995,214,15.33,10.73


In [15]:
run_sql("""
WITH order_delivery AS (
    SELECT
        order_id,
        CASE
            WHEN order_delivered_customer_date IS NULL THEN 'Not Delivered'
            WHEN CAST(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) AS INTEGER) > 0 THEN 'Delayed'
            ELSE 'On Time / Early'
        END AS delivery_status
    FROM orders
)

SELECT
    od.delivery_status,
    COUNT(DISTINCT od.order_id) AS total_orders,
    COUNT(DISTINCT r.review_id) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM order_delivery od
LEFT JOIN reviews r
    ON od.order_id = r.order_id
GROUP BY od.delivery_status
ORDER BY average_review_score DESC;
""")

,delivery_status,total_orders,total_reviews,average_review_score
0,On Time / Early,89941,89308,4.29
1,Delayed,6535,6390,2.27
2,Not Delivered,2965,2848,1.76


In [16]:
run_sql("""
SELECT
    review_score,
    COUNT(DISTINCT review_id) AS total_reviews,
    ROUND(
        COUNT(DISTINCT review_id) * 100.0 /
        (SELECT COUNT(DISTINCT review_id) FROM reviews),
        2
    ) AS review_percentage
FROM reviews
GROUP BY review_score
ORDER BY review_score;
""")

,review_score,total_reviews,review_percentage
0,1,11282,11.46
1,2,3114,3.16
2,3,8097,8.23
3,4,19007,19.31
4,5,56910,57.83


In [17]:
run_sql("""
SELECT
    SUBSTR(s.seller_id, 1, 8) AS seller_short_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items oi
LEFT JOIN sellers s
    ON oi.seller_id = s.seller_id
GROUP BY s.seller_id, s.seller_state
ORDER BY total_revenue DESC
LIMIT 10;
""")

,seller_short_id,seller_state,total_orders,total_items_sold,total_revenue,average_item_price
0,4869f7a5,SP,1132,1156,229472.63,198.51
1,53243585,BA,358,410,222776.05,543.36
2,4a3ca931,SP,1806,1987,200472.92,100.89
3,fa1c13f2,SP,585,586,194042.03,331.13
4,7c67e144,SP,982,1364,187923.89,137.77
5,7e93a43e,SP,336,340,176431.87,518.92
6,da8622b1,SP,1314,1551,160236.57,103.31
7,7a67c85e,SP,1160,1171,141745.53,121.05
8,1025f0e2,SP,915,1428,138968.55,97.32
9,955fee92,SP,1287,1499,135171.70,90.17


In [18]:
run_sql("""
SELECT
    s.seller_state,
    COUNT(DISTINCT oi.seller_id) AS total_sellers,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.freight_value), 2) AS total_freight,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items oi
LEFT JOIN sellers s
    ON oi.seller_id = s.seller_id
GROUP BY s.seller_state
ORDER BY total_revenue DESC
LIMIT 10;
""")

,seller_state,total_sellers,total_orders,total_items_sold,total_revenue,total_freight,average_item_price
0,SP,1849,70188,80342,8753396.21,1482487.67,108.95
1,PR,349,7673,8671,1261887.21,197013.52,145.53
2,MG,244,7930,8827,1011564.74,212595.06,114.60
3,RJ,171,4353,4818,843984.22,93829.90,175.17
4,SC,190,3667,4075,632426.07,106547.06,155.20
5,RS,129,1989,2199,378559.54,57243.09,172.15
6,BA,19,569,643,285561.56,19700.68,444.11
7,DF,30,824,899,97749.48,18494.06,108.73
8,PE,9,406,448,91493.85,12392.46,204.23
9,GO,40,463,520,66399.21,12565.50,127.69


In [19]:
run_sql("""
SELECT
    REPLACE(ct.product_category_name_english, '_', ' ') AS category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category_name
ORDER BY total_orders DESC
LIMIT 10;
""")

,category_name,total_orders,total_items_sold,total_revenue,average_item_price
0,bed bath table,9417,11115,1036988.68,93.30
1,health beauty,8836,9670,1258681.34,130.16
2,sports leisure,7720,8641,988048.97,114.34
3,computers accessories,6689,7827,911954.32,116.51
4,furniture decor,6449,8334,729762.49,87.56
5,housewares,5884,6964,632248.66,90.79
6,watches gifts,5624,5991,1205005.68,201.14
7,telephony,4199,4545,323667.53,71.21
8,auto,3897,4235,592720.11,139.96
9,toys,3886,4117,483946.60,117.55


In [20]:
run_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
         AND CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS INTEGER) > 0
        THEN o.order_id
    END) AS delayed_orders,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date IS NOT NULL
             AND CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS INTEGER) > 0
            THEN o.order_id
        END) * 100.0 / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2
    ) AS delayed_order_percentage

FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY delayed_orders DESC
LIMIT 10;
""")

,customer_state,total_orders,delayed_orders,delayed_order_percentage
0,SP,41746,1820,4.36
1,RJ,12852,1495,11.63
2,MG,11635,520,4.47
3,BA,3380,396,11.72
4,RS,5466,325,5.95
5,SC,3637,291,8.00
6,ES,2033,214,10.53
7,PR,5045,199,3.94
8,CE,1336,176,13.17
9,PE,1652,153,9.26


In [21]:
run_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,

    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN o.order_id
    END) AS delivered_orders,

    ROUND(
        AVG(
            CASE
                WHEN o.order_delivered_customer_date IS NOT NULL
                THEN CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS INTEGER)
            END
        ),
        2
    ) AS avg_delivery_days

FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC
LIMIT 10;
""")

,customer_state,total_orders,delivered_orders,avg_delivery_days
0,RR,46,41,28.98
1,AP,68,67,26.73
2,AM,148,145,25.99
3,AL,413,397,24.04
4,PA,975,946,23.32
5,MA,747,717,21.12
6,SE,350,335,21.03
7,CE,1336,1279,20.82
8,AC,81,80,20.64
9,PB,536,517,19.95
